# Case study: an epidemic and the hospital beds it needs {#sec-case-epidemic}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c14-epidemic.ipynb)

A new respiratory infection is spreading in a region of one million people. It is day 50 of the outbreak. Hospitals have admitted 90 patients so far, and the daily count has been doubling every five or six days. About 1,000 beds can be given over to these patients. The regional health authority asks three questions. When will the beds run out, if they do? How sure are we? And what would contact measures, or extra beds, buy us?

This chapter answers them with the whole modelling process of the book, compressed into one case: the problem and its reference mode, a dynamic hypothesis, a model built step by step, tests, an explanation of the behaviour, calibration to the admissions seen so far, and policy experiments. The outbreak and its data are **synthetic**: we made them with a short program in section 1, and they describe no real disease or place.

**Who it is for.** Readers who have worked through Parts I to IV and want to see the steps used together on a problem outside the waiting list. It also suits public health analysts who know epidemic models and want to see them as stock-and-flow models. **Who can skip.** Readers who already build and calibrate compartmental epidemic models can read sections 4 to 6 and the brief. **Prerequisites.** Chapter 5, Loops that grow, settle and tip (the Diffusion molecule and tipping points); chapter 6, Delays, pipelines and memory (staged delays); chapter 12, Testing your model; chapter 13, Data, calibration and uncertainty; and chapter 15, From model to decision. Least squares and the bootstrap come from chapter 11, Learning from data. **Time.** About three hours, with the exercises.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
from loopbricks import ai
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 14
rng = np.random.default_rng(SEED)

# Recorded AI replies (section 7): the local file if there is one,
# otherwise GitHub's copy
REPLAY = "data/ai/c14.json" if os.path.exists("data/ai/c14.json") else (
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/"
    "notebooks/data/ai/c14.json")

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

In [ ]:
#| fig-alt: "Stock-and-flow sketch. Susceptible, Exposed (two stages), Infectious and Recovered in a row; a branch from Infectious runs through three hospital stages to Recovered. Loop R links Infectious to Infection; loop B links Susceptible to Infection; Bed capacity is compared with the hospital stages."
from matplotlib.patches import FancyArrowPatch, Rectangle


def draw_epidemic(ax):
    """The chapter's model as a stock-and-flow sketch with its main loops."""
    def box(x, y, w, label, h=0.16):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, lw=2))
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
                fontsize=9)

    def arrow(a, b, pipe=False, rad=0.0):
        style = dict(arrowstyle="-|>", mutation_scale=16 if pipe else 11,
                     lw=3.5 if pipe else 1.2,
                     color="0.35" if pipe else "#0072B2",
                     connectionstyle=f"arc3,rad={rad}")
        ax.add_patch(FancyArrowPatch(a, b, **style))

    box(0.00, 0.62, 0.15, "Susceptible")
    box(0.24, 0.62, 0.17, "Exposed\n(2 stages)")
    box(0.50, 0.62, 0.15, "Infectious")
    box(0.84, 0.62, 0.15, "Recovered")
    box(0.45, 0.12, 0.26, "In hospital\n(3 stages)")
    box(0.81, 0.12, 0.18, "Bed capacity")
    arrow((0.15, 0.70), (0.24, 0.70), pipe=True)        # Infection
    arrow((0.41, 0.70), (0.50, 0.70), pipe=True)        # Becoming infectious
    arrow((0.65, 0.70), (0.84, 0.70), pipe=True)        # Recovering at home
    arrow((0.575, 0.62), (0.575, 0.28), pipe=True)      # Admissions
    arrow((0.68, 0.28), (0.90, 0.62), pipe=True)        # Discharges
    ax.text(0.195, 0.56, "Infection", ha="center", fontsize=8)
    ax.text(0.745, 0.74, "Recovering", ha="center", fontsize=8)
    ax.text(0.585, 0.43, "Admissions", fontsize=8)
    ax.text(0.83, 0.40, "Discharges", fontsize=8)
    arrow((0.56, 0.78), (0.195, 0.72), rad=0.4)         # Infectious -> inf.
    arrow((0.07, 0.78), (0.17, 0.72), rad=-0.6)         # Susceptible -> inf.
    ax.add_patch(FancyArrowPatch((0.71, 0.18), (0.81, 0.18),
                                 arrowstyle="<|-|>", mutation_scale=10,
                                 lw=1.2, ls="--", color="#0072B2"))
    ax.text(0.76, 0.08, "compare", fontsize=8, ha="center",
            color="#0072B2")
    ax.text(0.38, 0.93, "R", fontsize=14, weight="bold", ha="center")
    ax.text(0.10, 0.93, "B", fontsize=14, weight="bold", ha="center")
    ax.text(0.12, 0.30, "Contact measures\nreduce Infection", fontsize=8,
            ha="center")
    arrow((0.15, 0.40), (0.19, 0.68))
    ax.set_xlim(-0.02, 1.01)
    ax.set_ylim(0.05, 1.0)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(7.5, 3.4))
draw_epidemic(ax)
ax.set_title("An epidemic and the hospital beds it needs")
plt.show()

Figure: The chapter's model as a stock-and-flow sketch. Thick arrows move people; thin arrows carry information. R marks the reinforcing loop of contagion (more infectious people, more infections) and B the balancing loop of depletion (fewer susceptible people, fewer infections).

**Learning outcomes.** By the end of the chapter you can:

1. **describe** a reference mode from noisy early counts and **estimate** a doubling time from them (Try it 1);
2. **state** a dynamic hypothesis and model boundary for an epidemic and **relate** it to the Diffusion molecule (Try it 2);
3. **build** an epidemic model with staged delays and a hospital stay, and **verify** it with conservation, extreme-condition and time-step tests (Try it 3);
4. **explain** why the epidemic turns when susceptibles fall below $1/R_0$ of the population, and how delays shift the peak in beds (Try it 4);
5. **calibrate** the transmission rate to early admissions and **quantify** how uncertain the peak forecast is (Try it 5);
6. **compare** contact measures and surge capacity, and **judge** which policies stay good when $R_0$ and the hospital share are uncertain (Try it 6);
7. **use** a generative draft, a reasoning review, an agent and a drafted brief at each stage of the case, and **check** every reply by code (Try it 7).

## 1. The problem and the reference mode

The authority's data are a single series: patients admitted to hospital each day. Like the waiting list of chapter 13, the series is synthetic. A short numpy program, `make_outbreak`, writes down an outbreak whose settings we know, and random noise is added on top. Because we know the truth, we can later check whether calibration recovers it.

The program follows one person's path: infected, then two latent stages (infected but not yet infectious), then one infectious stage, after which 2% go to hospital. Each person infects others at a **transmission rate** of 0.5 a day while infectious, and stays infectious for 5 days on average. Two kinds of noise are laid over the expected counts. A **Poisson** draw (`rng.poisson`) gives the scatter of counting whole people when each day's count is a matter of chance. A **lognormal** factor with a standard deviation of 0.2 on the log scale (`rng.lognormal`) adds day-to-day swings that a Poisson draw alone would understate, such as weekends and reporting delays. The seed is fixed, so the data are the same every time.

In the output below, the first admission came on day 16. On day 50 there were 13, and 90 people have been admitted in all. Daily counts this small are noisy: a single unusual day moves the picture. Plotted on a log scale, though, steady exponential growth becomes a straight line, and a least-squares line (chapter 11) through the logarithms of the counts estimates its slope.

In [ ]:
def make_outbreak(days=120, beta=0.5, exposed_0=20, N=1_000_000,
                  latent=4.0, infectious=5.0, share=0.02, step=0.1):
    """Expected admissions on each day 1..days of a synthetic outbreak.

    A plain numpy program, written apart from the model of section 3:
    two latent stages, one infectious stage, a share admitted to hospital.
    """
    S, E1, E2, I, admitted = N - exposed_0, exposed_0, 0.0, 0.0, 0.0
    per_day = round(1 / step)
    totals = [0.0]                         # admissions so far, each midnight
    for k in range(days * per_day):
        infection = beta * S * I / N
        e_out, e2_out = E1 / (latent / 2), E2 / (latent / 2)
        leaving = I / infectious
        S -= step * infection
        E1 += step * (infection - e_out)
        E2 += step * (e_out - e2_out)
        I += step * (e2_out - leaving)
        admitted += step * share * leaving
        if (k + 1) % per_day == 0:
            totals.append(admitted)
    return np.diff(totals)                 # admissions during each day


TRUE = {"transmission_rate": 0.5, "initial_exposed": 20}   # R0 = 2.5
expected = make_outbreak()
noise = rng.lognormal(0, 0.2, size=expected.size)   # day-to-day swings
obs = pd.DataFrame({"day": np.arange(1, 121),
                    "admissions": rng.poisson(expected * noise)})
TODAY = 50                                      # what the analyst has seen
seen = obs[obs.day <= TODAY]
print(seen.tail(7).to_string(index=False))
print(f"admitted by day {TODAY}: {seen.admissions.sum()}; first admission "
      f"on day {seen.day[seen.admissions > 0].iloc[0]}")

In [ ]:
#| fig-alt: "Two panels of daily admissions over the first 50 days. Left, on a linear scale, the bars stay near zero for weeks and then climb. Right, on a log scale, the points scatter about a rising straight line."
recent = seen[(seen.day > 30) & (seen.admissions > 0)]
slope, icept = np.polyfit(recent.day, np.log(recent.admissions), 1)
print(f"growth rate {slope:.3f} a day from days 31 to {TODAY}: "
      f"doubling every {np.log(2) / slope:.1f} days")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
a1.bar(seen.day, seen.admissions, color="#0072B2")
a1.set(title="Daily admissions so far", xlabel="time (days)",
       ylabel="admissions (people/day)")
pos = seen[seen.admissions > 0]
a2.semilogy(pos.day, pos.admissions, "o", color="#0072B2", ms=4,  # log y
            label="data (days with admissions)")
a2.semilogy(recent.day, np.exp(icept + slope * recent.day), color="#D55E00",
            ls="--", label="least-squares line, days 31 on")
a2.set(title="The same on a log scale", xlabel="time (days)",
       ylabel="admissions (people/day, log scale)")
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: Daily admissions in the first 50 days, on a linear scale (left) and a log scale (right) with a least-squares line through days 31 to 50. On the linear scale the outbreak looks like it has only just started; on the log scale the growth has been steady for weeks.

The slope is 0.127 a day: admissions double every 5.5 days. This is the **reference mode** as far as the data go: exponential growth. Nobody expects it to go on for ever. The pattern we fear, and the one we must explain, is the familiar one for epidemics:

- cumulative infections and admissions trace an **S-shaped curve**, rising slowly, then steeply, then levelling off;
- daily admissions rise to a **peak** and fall away;
- beds occupied follow the admissions with a lag, and may rise **above the 1,000 beds** available for weeks.

The decision maker needs the timing and height of that peak in beds, and the range of both.

**Try it 1.**

- *Check.* What is the largest daily count so far, and on which day?
- *Apply.* Fit the line from day 21 instead of day 31. What doubling time do you get, and why does it differ?
- *Extend.* Suppose admissions kept doubling at the fitted rate and each patient stayed 10 days, so beds occupied were about ten times daily admissions. On which day would 1,000 beds be full? Keep the answer; section 3 compares it with the model.

<details><summary>Worked answer</summary>

The largest count is 13, on day 50. From day 21 the doubling time is 6.2 days: the extra early points are single admissions on a few scattered days (zero days are left out, since the logarithm of zero is not defined), and they pull the line flatter. Early counts this small carry little information, which section 5 measures properly. The straight-line forecast puts admissions at 10.9 a day on day 50 and at 100 a day, so 1,000 beds full, around day 67.
</details>

In [ ]:
print(f"check: largest day so far {seen.admissions.max()} on day "
      f"{seen.day[seen.admissions.idxmax()]}; total {seen.admissions.sum()}")
early = seen[(seen.day > 20) & (seen.admissions > 0)]
s2, _ = np.polyfit(early.day, np.log(early.admissions), 1)
print(f"apply: from day 21, doubling every {np.log(2) / s2:.1f} days")
# extend: beds ~ admissions x 10-day stay; full when admissions reach 100/day
level_today = np.exp(icept + slope * TODAY)
days_to_full = np.log(100 / level_today) / slope
print(f"extend: trend level today {level_today:.1f}/day; 1,000 beds full "
      f"about day {TODAY + days_to_full:.0f}")

## 2. Dynamic hypothesis and boundary

Why should an epidemic rise and then fall? The **dynamic hypothesis** is a statement of the feedback structure that we think produces the reference mode. It can be put in four parts.

- **Contagion (reinforcing loop R).** Infectious people infect susceptible people, who become infectious in turn. While nearly everyone is susceptible, the number of infections grows in proportion to itself, which is exponential growth.
- **Depletion (balancing loop B).** Each infection removes one person from the susceptible pool, so every infectious person finds fewer people to infect. Eventually the pool is too small to sustain growth.
- **Recovery (balancing).** Infectious people stop being infectious after a few days. This is a drain on the infectious stock, like the Residence Time molecule of chapter 5.
- **Delays.** An infected person is not infectious at once (the latent period), and a patient occupies a bed for days (the length of stay). These delays separate the peak in infections from the peak in beds.

The **Diffusion** molecule (`diffusion`), after Hines, *Molecules of Structure* [@hines_molecules], and met in chapter 5, already contains the first two loops. It is the structure of @bass1969, where people adopt through word of mouth with adopters. Switch off the innovation channel, start with one adopter, and read Adopters as "infected": it is an epidemic in which nobody ever recovers.

In [ ]:
#| fig-alt: "Two panels from the Diffusion molecule run as an epidemic with no recovery. Left, infected people follow an S-curve from 1 to 1,000. Right, new infections a week rise to a single peak and fall."
dif = lb.molecule("diffusion")
print(dif.equations())
# an epidemic with no recovery: one case, no outside source, q = 50% a week
si = dif.with_init("Adopters", 1).with_init("Potential", 999).run(
    innovation_p=0, imitation_q=50)
top = si["Adoption"].idxmax()
print(f"\nnew infections peak in week {top} at {si['Adoption'].max():.1f} a "
      f"week, when {si.loc[top, 'Adopters']:.0f} of 1,000 are infected; "
      f"{si['Adopters'].iloc[-1]:.0f} infected by week 60")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
si["Adopters"].plot(ax=a1, label="infected so far")
a1.set(title="Infected (Adopters)", xlabel="time (weeks)",
       ylabel="people")
si["Adoption"].plot(ax=a2, color="#D55E00", ls="--")
a2.set(title="New infections (Adoption)", xlabel="time (weeks)",
       ylabel="infections (people/week)")
plt.tight_layout()
plt.show()

Figure: The Diffusion molecule run as an epidemic with no recovery among 1,000 people, starting from one case: infections follow an S-curve (left) and new infections rise to one peak and fall (right). Notice that the peak comes when half the population is infected.

New infections peak in week 14.1, when 501 of the 1,000 people are infected, and everyone is infected by week 60. Without recovery the epidemic turns at half the population, as for any logistic curve (chapter 5). With recovery, the turn comes earlier, and not everyone is infected.

The key number is the **basic reproduction number** $R_0$: how many people one infectious person infects in a population where everyone is susceptible. With a transmission rate $\beta$ (infections per infectious person per day, when all contacts are susceptible) and an average infectious period $D$,

$$
R_0 = \beta D, \qquad R_e(t) = R_0 \, \frac{S(t)}{N},
$$

where the **effective reproduction number** $R_e$ counts the infections per case when only a fraction $S/N$ of contacts are still susceptible. The epidemic grows while $R_e > 1$ and shrinks once $R_e < 1$, so it turns when $S/N$ falls below $1/R_0$. This is the tipping point of chapter 5, where the dropout version of Diffusion survived only when each adopter recruited more than one other.

The **model boundary** says what the model explains, what it takes as given, and what it leaves out.

| Endogenous (explained by the model) | Exogenous (set by us) | Excluded |
|---|---|---|
| Susceptible, exposed, infectious, recovered people | Population, 1,000,000 people | Births, deaths from other causes, migration |
| Infections, admissions, discharges | Transmission rate, latent and infectious periods | Loss of immunity, vaccination |
| Beds occupied, effective $R$ | Hospital share, length of stay | Age groups, places, contact networks |
| Patients without a bed | Bed capacity, surge beds, contact measures and their timing | Deaths in hospital, and any extra deaths when beds run out |
|  |  | People changing their behaviour on their own as cases rise |

The last row matters most for policy. People often cut their contacts when they hear that hospitals are full, a balancing loop this model leaves out; we treat contact measures as decisions, not as reactions. Going further returns to it.

**Try it 2.**

- *Check.* Run the epidemic without recovery with `imitation_q` of 25 and 100. How does the timing of the peak change?
- *Apply.* For $R_0$ of 1.5, 2.5 and 4, below what share of the population must susceptibles fall before the epidemic turns?
- *Extend.* Name one excluded item in the boundary table that would change the peak in beds, and say which way.

<details><summary>Worked answer</summary>

With `imitation_q=25` the peak moves to week 27.9; with 100 it comes in week 7.2. Doubling the contact rate halves the time scale, but everyone is still infected in the end. The epidemic turns when susceptibles fall below 67%, 40% and 25% of the population. One answer to the extension: people cutting their contacts on their own as cases rise would add a balancing loop, lower and widen the peak, and make our unmitigated runs too pessimistic.
</details>

In [ ]:
for q in (25, 50, 100):
    run = dif.with_init("Adopters", 1).with_init("Potential", 999).run(
        T=120, innovation_p=0, imitation_q=q)
    print(f"q = {q:>3}% a week: peak in week {run['Adoption'].idxmax():5.1f},"
          f" {run['Adopters'].iloc[-1]:.0f} infected by week 120")
for r0 in (1.5, 2.5, 4):
    print(f"R0 = {r0}: the epidemic turns when susceptibles fall below "
          f"{1 / r0:.0%} of the population")

## 3. The model in code, step by step

We build the model as a spec, with the same helper functions as chapter 5 (`stock`, `flow` and `var`), and pass it to `lb.Model.from_spec`. Appendix B (@sec-package) documents the spec format and the equation language.

**The epidemic.** The infection flow is the imitation term of Diffusion, with Infectious in the place of Adopters and a transmission rate in the place of $q/100$:

$$
\text{Infection} = m(t)\,\beta\,\frac{S\,I}{N},
$$

where $m(t)$ is a **contact multiplier**: 1 normally, and $1 - c$ while contact measures cut contacts by a fraction $c$. Two `STEP` functions switch the measures on at `Measures start` and off again `Measures length` days later. The latent period is a two-stage delay, the Material Delay of chapter 6 with two stages instead of three, so that few people become infectious in the first day or two. The infectious period has a single stage, which makes the turning point exact, as section 4 shows. A `Hospital share` of those leaving the infectious stage are admitted; the rest recover at home.

In [ ]:
def stock(i, name, init, units, non_negative=True):
    """A stock for a model spec."""
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": non_negative, "x": 0, "y": 0}


def flow(i, name, eq, units, src=None, dst=None):
    """A flow for a model spec, from stock src to stock dst."""
    f = {"id": i, "name": name, "eq": eq, "units": units, "x": 0, "y": 0}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units):
    """A variable (a constant or an equation) for a model spec."""
    return {"id": i, "name": name, "eq": str(eq), "units": units,
            "x": 0, "y": 0}


PEOPLE, RATE = "people", "people/day"
SEIR_STOCKS = [
    stock("s", "Susceptible", "Population - Initial_exposed", PEOPLE),
    stock("e1", "Exposed 1", "Initial_exposed", PEOPLE),
    stock("e2", "Exposed 2", 0, PEOPLE),
    stock("i", "Infectious", 0, PEOPLE),
    stock("r", "Recovered", 0, PEOPLE)]
SEIR_FLOWS = [
    flow("inf", "Infection", "Contact_multiplier * Transmission_rate"
         " * Susceptible * Infectious / Population", RATE, "s", "e1"),
    flow("inc", "Incubating", "Exposed_1 / (Latent_period / 2)", RATE,
         "e1", "e2"),
    flow("ons", "Becoming infectious", "Exposed_2 / (Latent_period / 2)",
         RATE, "e2", "i"),
    flow("rec", "Recovering at home",
         "(1 - Hospital_share) * Infectious / Infectious_period", RATE,
         "i", "r")]
SEIR_VARS = [
    var("N", "Population", 1000000, PEOPLE),
    var("e0", "Initial exposed", 20, PEOPLE),
    var("beta", "Transmission rate", 0.5, "1/day"),
    var("lat", "Latent period", 4, "days"),
    var("dur", "Infectious period", 5, "days"),
    var("hs", "Hospital share", 0.02, "fraction"),
    var("R0", "R0", "Transmission_rate * Infectious_period", "dmnl"),
    var("cm", "Contact multiplier", "1 - Contact_reduction"
        " * (STEP(1, Measures_start) - STEP(1, Measures_start"
        " + Measures_length))", "fraction"),
    var("cr", "Contact reduction", 0, "fraction"),
    var("ms", "Measures start", 50, "day"),
    var("ml", "Measures length", 60, "days")]

**The hospital.** Patients stay 10 days on average, in a chain of three stages: the **Cascaded Levels** molecule (`cascade`), after Hines, *Molecules of Structure* [@hines_molecules], which chapter 6 used for the referral backlog. A chain like this is an **ageing chain**: people move through the stages in order, as in the **Aging Chain** molecule (`aging`), also after Hines [@hines_molecules]. Three stages give a spread of stays without many very short ones. A separate stock, `Admitted so far`, counts every admission and never empties; its daily differences are what the hospital reports. `RAMP` builds surge beds up over `Surge build time` days from `Surge start`, and `MIN` stops at the number planned. `Patients without a bed` measures the danger: how far occupancy runs above capacity. After building the model, we check that three stages of 10/3 days give sensible stays. The Cascaded Levels molecule starts with 60 people in its first stage, so it shows a batch of 60 patients admitted together.

In [ ]:
STAGE = "(Length_of_stay / 3)"
HOSP_STOCKS = [stock(f"h{k}", f"In hospital {k}", 0, PEOPLE)
               for k in (1, 2, 3)]
HOSP_STOCKS.append(stock("adm_total", "Admitted so far", 0, PEOPLE))
HOSP_FLOWS = [
    flow("adm", "Admissions",
         "Hospital_share * Infectious / Infectious_period", RATE, "i", "h1"),
    flow("h12", "Stay 1 to 2", f"In_hospital_1 / {STAGE}", RATE, "h1", "h2"),
    flow("h23", "Stay 2 to 3", f"In_hospital_2 / {STAGE}", RATE, "h2", "h3"),
    flow("dis", "Discharges", f"In_hospital_3 / {STAGE}", RATE, "h3", "r"),
    flow("cnt", "Counting admissions", "Admissions", RATE, dst="adm_total")]
HOSP_VARS = [
    var("los", "Length of stay", 10, "days"),
    var("occ", "Beds occupied",
        "In_hospital_1 + In_hospital_2 + In_hospital_3", "beds"),
    var("cap", "Bed capacity", "Base_beds + Surge_beds"
        " * MIN(1, RAMP(1 / Surge_build_time, Surge_start))", "beds"),
    var("bb", "Base beds", 1000, "beds"),
    var("sb", "Surge beds", 0, "beds"),
    var("ss", "Surge start", 60, "day"),
    var("sbt", "Surge build time", 14, "days"),
    var("short", "Patients without a bed",
        "MAX(0, Beds_occupied - Bed_capacity)", PEOPLE),
    var("re", "Effective R", "R0 * Contact_multiplier * Susceptible"
        " / Population", "dmnl")]

epi = lb.Model.from_spec({
    "name": "Epidemic and hospital beds", "timeUnit": "day",
    "T": 250, "dt": 0.25,
    "stocks": SEIR_STOCKS + HOSP_STOCKS,
    "flows": SEIR_FLOWS + HOSP_FLOWS,
    "vars": SEIR_VARS + HOSP_VARS})
print(epi.equations())

stay = lb.molecule("cascade").run(stage_delay=10 / 3, T=40)
out = stay["Outflow"]
mean_stay = (out.index * out).sum() * 0.1 / 60
print(f"60 patients admitted at once: discharges peak on day "
      f"{out.idxmax()}, mean stay {mean_stay:.2f} days, "
      f"{100 * out.loc[:3].sum() * 0.1 / 60:.1f}% out within 3 days")

Discharges peak on day 6.6 (two-thirds of the 10-day stay, as chapter 6 predicts for three stages), the mean stay is 9.88 days (just under 10 because the run stops at day 40), and only 6.5% of patients leave within 3 days. That is a plausible shape for a respiratory admission. Now run the whole model with the true settings of section 1 and no measures.

In [ ]:
#| fig-alt: "Two panels over 250 days. Left: the share of people susceptible falls in an S from 100% to about 11%, those ever infected rise in an S, and those currently infected form a hump. Right: beds occupied rise to a hump near 4,800, far above a dotted capacity line at 1,000."
base = epi.run()
infected = base[["Exposed 1", "Exposed 2", "Infectious"]].sum(axis=1)
beds = base["Beds occupied"]
over = beds[beds > 1000]
print(f"ever infected: {1e6 - base['Susceptible'].iloc[-1]:,.0f}; "
      f"admitted: {base['Admitted so far'].iloc[-1]:,.0f}")
print(f"beds: peak {beds.max():,.0f} on day {beds.idxmax()}; over 1,000 "
      f"from day {over.index[0]} to day {over.index[-1]}")
print(f"patient-days without a bed: "
      f"{base['Patients without a bed'].sum() * 0.25:,.0f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
(100 * base["Susceptible"] / 1e6).plot(ax=a1, label="susceptible")
(100 * (1 - base["Susceptible"] / 1e6)).plot(ax=a1, label="ever infected")
(100 * infected / 1e6).plot(ax=a1, label="infected now (E + I)")
a1.set(title="The epidemic", xlabel="time (days)",
       ylabel="share of population (%)")
a1.legend(fontsize=8)
beds.plot(ax=a2, label="beds occupied")
a2.axhline(1000, color="grey", ls=":", lw=1, label="bed capacity")
a2.set(title="Hospital beds", xlabel="time (days)", ylabel="beds")
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: The unmitigated epidemic: the share still susceptible falls in an S, the share ever infected rises in an S and the share infected now is a hump (left). Notice that beds occupied rise to almost five times the capacity of 1,000 beds (right).

Without measures, 894,421 people are infected, 89% of the region, and 17,888 are admitted. Beds occupied peak at 4,809 on day 100.5, and occupancy is above 1,000 from day 75.25 to day 127.75. Adding up the patients over capacity each day gives 107,563 **patient-days without a bed**: the number of days, summed over patients, that someone who needed a bed did not have one. It is our main measure of harm.

Compare with Try it 1: the straight-line forecast filled the beds on day 67, eight days earlier than the model. Beds lag admissions, because a bed fills on admission and empties only days later, and in fast growth most patients have only just arrived. Ten times today's admissions overstates occupancy while the epidemic is growing fast.

### Tests before trust

Chapter 12 asks two questions of every model: is the code right, and is the model right? Four quick tests answer the first.

1. **Conservation.** Every person is in exactly one of the eight living stocks (`Admitted so far` is a counter, not a place), so their sum must stay at 1,000,000.
2. **Extreme conditions.** With no transmission, only the 20 initial cases are ever infected. With nobody exposed, nothing happens. With a hospital share of 0, no beds are used; with 1, everyone infected is admitted.
3. **Time step.** Halving `dt` should change the answer little.
4. **Against the generator.** Run with the generator's own step of 0.1 days, the model must reproduce the expected admissions of section 1 exactly.

In [ ]:
living = ["Susceptible", "Exposed 1", "Exposed 2", "Infectious",
          "In hospital 1", "In hospital 2", "In hospital 3", "Recovered"]
drift = (base[living].sum(axis=1) - 1e6).abs().max()
print(f"conservation: largest drift from 1,000,000 people: {drift:.2e}")
assert drift < 1e-3

extremes = {"no transmission": dict(transmission_rate=0),
            "nobody exposed": dict(initial_exposed=0),
            "nobody needs a bed": dict(hospital_share=0),
            "everyone needs a bed": dict(hospital_share=1)}
for label, kw in extremes.items():
    d = epi.run(**kw)
    ever = 1e6 - d["Susceptible"].iloc[-1]
    print(f"{label:<22}: ever infected {ever:>9,.1f}"
          f", admitted {d['Admitted so far'].iloc[-1]:>9,.1f}, "
          f"peak beds {d['Beds occupied'].max():>8,.1f}")

for dt in (0.5, 0.25, 0.125, 0.0625):
    b = epi.run(dt=dt)["Beds occupied"]
    print(f"dt = {dt:<6}: peak beds {b.max():7,.1f} on day {b.idxmax()}")

# the data generator of section 1 steps 0.1 days at a time; so must we
same = epi.run(dt=0.1, T=120)["Admitted so far"].loc[np.arange(0.0, 121)]
print(f"model against the data generator, days 1 to 120: largest gap "
      f"{np.abs(np.diff(same) - expected).max():.1e} admissions a day")

Everyone stays accounted for: the sum of the stocks drifts from 1,000,000 by about $10^{-9}$, which is rounding. The extreme runs behave as they should: 20 people infected and 0.4 admitted without transmission, nothing at all without initial cases, no beds with a share of 0, and every infected person admitted with a share of 1. The model and the generator agree to within $2 \times 10^{-12}$ admissions a day, so the code does what section 1's program does.

The time-step test needs more care. The peak in beds is 4,870.9, 4,808.7, 4,778.4 and 4,763.4 as `dt` halves from 0.5 to 0.0625 days: each halving changes it by about half as much as the last, the signature of Euler's method (chapter 1). Our step of 0.25 days is about 1% above the limit, close enough for a forecast whose uncertainty, as section 5 shows, is far larger. The latent stages last 2 days each, so 0.25 is an eighth of the shortest time constant.

**Try it 3.**

- *Check.* With `hospital_share=1` and a run of 400 days, is everyone infected admitted?
- *Apply.* Run lengths of stay of 7, 10 and 14 days. How do the peak in beds and its timing change?
- *Extend.* Run the base case with `method="rk4"`, fourth-order Runge-Kutta. Compare its peak with the Euler runs at smaller and smaller `dt`.

<details><summary>Worked answer</summary>

Yes: 894,420.7 admitted and 894,420.7 ever infected. Stays of 7, 10 and 14 days give peaks of 3,492, 4,809 and 6,355 beds, on days 98.75, 100.5 and 102.5. The peak in beds scales almost in proportion to the stay, and comes a little later with longer stays. Runge-Kutta gives 4,748.5 on day 99.25, which is where the Euler peaks were heading: 4,763.4 at the smallest step, falling by half as much at each halving.
</details>

In [ ]:
d = epi.run(hospital_share=1, T=400)
print(f"check: hospital share 1, everyone admitted equals everyone "
      f"infected? {d['Admitted so far'].iloc[-1]:,.1f} and "
      f"{1e6 - d['Susceptible'].iloc[-1]:,.1f}")
for los in (7, 10, 14):
    b = epi.run(length_of_stay=los)["Beds occupied"]
    print(f"apply: stay {los:>2} days, peak beds {b.max():6,.0f} on day "
          f"{b.idxmax()}")
rk = epi.run(method="rk4")["Beds occupied"]
print(f"extend: rk4 peak {rk.max():,.1f} on day {rk.idxmax()}")

## 4. Explaining the behaviour

Why does the epidemic turn when it does, and why do the beds peak later? Write $E$ for everyone exposed and $I$ for the infectious. With one infectious stage, people enter $E + I$ by infection and leave by the end of their infectious period, so

$$
\frac{d(E + I)}{dt} = \beta \frac{S I}{N} - \frac{I}{D} = \frac{I}{D}\left(R_0 \frac{S}{N} - 1\right) = \frac{I}{D}\,(R_e - 1).
$$

The number of people currently infected grows while $R_e > 1$ and falls once $R_e < 1$. It peaks exactly when $S/N = 1/R_0$: with $R_0 = 2.5$, when 400,000 people are still susceptible. That is the turning point the decision maker should watch, and the plot shows it in the model.

In [ ]:
#| fig-alt: "Effective R falls from 2.5 and crosses a dotted line at 1 around day 91. Three humps, each scaled to its own peak, follow: people currently infected peak at that moment, admissions about three days later and beds occupied about six days after that."
re = base["Effective R"]
turn = re.index[re < 1][0]
peaks = {"infection": base["Infection"].idxmax(),
         "infected now (E + I)": infected.idxmax(),
         "admissions": base["Admissions"].idxmax(),
         "beds occupied": beds.idxmax()}
print(f"effective R falls below 1 on day {turn}, with "
      f"{base.loc[turn, 'Susceptible']:,.0f} susceptible (N / R0 = 400,000)")
for k, v in peaks.items():
    print(f"  {k:<22} peaks on day {v}")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(re.index, re, label="effective R")
ax.axhline(1, color="grey", ls=":", lw=1)
ax.set(title="The turn comes when effective R falls below 1",
       xlabel="time (days)", ylabel="effective R (dimensionless)")
ax2 = ax.twinx()                      # a second y axis on the right
for col, ls, (k, s) in zip(["#D55E00", "#009E73", "#CC79A7"],
                           ["--", ":", "-."],
                           [("infected now (E + I)", infected),
                            ("admissions", base["Admissions"]),
                            ("beds occupied", beds)]):
    ax2.plot(s.index, 100 * s / s.max(), color=col, ls=ls, label=k)
ax2.set_ylabel("share of its own peak (%)")
ax2.grid(False)
lines = ax.get_lines()[:1] + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], fontsize=8,
          loc="upper right")
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()

R0 = 2.5
z = 0.5                         # share ever infected, a first guess
for _ in range(100):            # iterate z = 1 - exp(-R0 z) to its fixed point
    z = 1 - np.exp(-R0 * z)
print(f"final size equation: {z:.1%} infected; model "
      f"{1 - base['Susceptible'].iloc[-1] / 1e6:.1%}; herd immunity "
      f"threshold 1 - 1/R0 = {1 - 1 / R0:.0%}")

Figure: Effective $R$ (left axis) and three humps scaled to their own peaks (right axis). People infected now peak when effective $R$ crosses 1; admissions and beds follow in that order, each a few days later.

Effective $R$ falls below 1 on day 91.5, with 398,853 people susceptible, against the theory's 400,000 (Euler steps of a quarter-day explain the difference). People infected now peak on that day. New infections peaked earlier, on day 86, because they depend on $S \times I$ and $S$ was already falling fast. Admissions peak on day 94.25, three days later, because admission waits for the end of the infectious period. Beds peak on day 100.5, six days after admissions, because a bed stays full until discharge: occupancy keeps rising while admissions exceed discharges, even after admissions have turned.

This is shifting loop dominance (chapter 5). Up to about day 91 the reinforcing loop of contagion dominates; afterwards depletion does. Nobody acted: the epidemic turned because it had used up its own fuel. But it does not stop at the turning point. People already exposed or infectious go on to infect others, so the epidemic **overshoots** the **herd immunity threshold** $1 - 1/R_0$, the share that must be immune for $R_e$ to fall below 1. The final share infected $z$ solves the classic equation of Kermack and McKendrick [@kermack1927],

$$
z = 1 - e^{-R_0 z},
$$

which we can solve by iterating it from a first guess.

The equation gives 89.3% infected and the model 89.4%, against a herd immunity threshold of 60%. An unmitigated epidemic with $R_0 = 2.5$ infects about 29 percentage points more people than the threshold requires. Measures that slow the epidemic near its peak can reduce that overshoot, which section 6 uses.

**Try it 4.**

- *Check.* What is the herd immunity threshold for $R_0 = 2.5$, and what share of the population is infected beyond it in the base run?
- *Apply.* Run transmission rates of 0.4 and 0.6 a day ($R_0$ of 2 and 3). On which day do people infected now peak, and how many are susceptible then? Compare with $N/R_0$.
- *Extend.* Run lengths of stay of 5 and 20 days. How long after admissions do beds peak? Explain with the stock of beds and its two flows.

<details><summary>Worked answer</summary>

The threshold is 60%; the base run infects 89.4%, so 29.4 percentage points more. With $R_0 = 2$, $E + I$ peaks on day 120.5 with 497,517 susceptible ($N/R_0 = 500{,}000$); with $R_0 = 3$, on day 76.0 with 328,649 (333,333). Beds peak 3.25 days after admissions with a 5-day stay and 10.75 days after with a 20-day stay. Beds peak when discharges catch up with admissions; the longer the stay, the longer discharges lag behind, so the later the peak in beds.
</details>

In [ ]:
for beta in (0.4, 0.6):
    d = epi.run(transmission_rate=beta, T=300)
    e_i = d[["Exposed 1", "Exposed 2", "Infectious"]].sum(axis=1)
    print(f"R0 {beta * 5:.1f}: E + I peaks on day {e_i.idxmax()}, with "
          f"{d.loc[e_i.idxmax(), 'Susceptible']:,.0f} susceptible "
          f"(N / R0 = {1e6 / (beta * 5):,.0f})")
for los in (5, 20):
    b = epi.run(length_of_stay=los)["Beds occupied"]
    print(f"stay {los} days: beds peak on day {b.idxmax()}, "
          f"{b.idxmax() - base['Admissions'].idxmax():.2f} days after "
          f"admissions")

## 5. Calibrating to early admissions: how sure is the peak

It is day 50. The authority wants a forecast of the peak in beds now, not when the data have made it obvious. We calibrate the model to the admissions seen so far, as in chapter 13, and then ask how much the forecast could change with a different draw of noise.

Two parameters are uncertain: the **transmission rate** and the number of **initial exposed** people, which sets when the outbreak effectively started. The latent and infectious periods and the hospital share we take as known from clinical studies (Try it 5 relaxes the last). Two parameters suit a grid search: `run_batch` runs every combination at once (chapter 5), here 71 transmission rates from 0.30 to 1.00 a day and 41 initial sizes from 0.5 to 300 people, spaced evenly on a log scale by `np.geomspace`. The payoff is the sum of squared errors between modelled and observed daily admissions, chapter 11's least squares.

In [ ]:
B = np.arange(0.30, 1.0001, 0.01)              # transmission rate, 1/day
E0 = np.geomspace(0.5, 300, 41)                 # initial exposed, people
BB, EE = np.meshgrid(B, E0)
grid = pd.DataFrame({"transmission_rate": BB.ravel(),
                     "initial_exposed": EE.ravel()})
runs = epi.run_batch(grid, T=250,
                     include=["Admitted so far", "Beds occupied"])
cum = runs[runs.variable == "Admitted so far"].pivot(
    index="time", columns="run", values="value")
occ = runs[runs.variable == "Beds occupied"].pivot(
    index="time", columns="run", values="value")
DAILY = np.diff(cum.loc[np.arange(0.0, 121)].to_numpy(), axis=0)  # day x run
PEAK, PEAK_DAY = occ.max().to_numpy(), occ.idxmax().to_numpy()
print(f"{len(grid):,} runs; DAILY has shape {DAILY.shape}")


def fit(y, upto):
    """Index of the grid run with least squared error on days 1..upto."""
    sse = ((DAILY[:upto] - y[:upto, None]) ** 2).sum(axis=0)
    return int(np.argmin(sse))


y = obs.admissions.to_numpy(dtype=float)
best = fit(y, TODAY)
print(f"fit to day {TODAY}: transmission rate {BB.ravel()[best]:.3f} a day "
      f"(R0 {5 * BB.ravel()[best]:.2f}), initial exposed "
      f"{EE.ravel()[best]:.1f}; peak beds {PEAK[best]:,.0f} on day "
      f"{PEAK_DAY[best]}")


def peak_spread(upto, n_boot=300):
    """Fit to days 1..upto, then refit to n_boot synthetic data sets."""
    j = fit(y, upto)
    mean = DAILY[:, j]
    sims = rng.poisson(mean[:, None]
                       * rng.lognormal(0, 0.2, (len(mean), n_boot)))
    D, Y = DAILY[:upto], sims[:upto]
    # sum (d - y)^2 = sum d^2 - 2 sum d y + sum y^2; the last term is the
    # same for every grid run, so it cannot change which run is best
    sse = (D ** 2).sum(axis=0)[:, None] - 2 * D.T @ Y   # runs x data sets
    picks = sse.argmin(axis=0)               # best grid run for each set
    return j, picks


rows, picks_by_cut = [], {}
for upto in (45, 50, 60, 70, 80, 90):
    j, picks = peak_spread(upto)
    picks_by_cut[upto] = picks
    lo, hi = np.percentile(PEAK[picks], [5, 95])
    dlo, dhi = np.percentile(PEAK_DAY[picks], [5, 95])
    rows.append({"data to day": upto, "R0": 5 * BB.ravel()[j],
                 "peak beds": PEAK[j], "5%": lo, "95%": hi,
                 "peak day": PEAK_DAY[j], "day 5%": dlo, "day 95%": dhi})
table = pd.DataFrame(rows).set_index("data to day")
print(table.round(2).to_string())
print(f"truth: peak {beds.max():,.0f} beds on day {beds.idxmax()}")

At day 50 the best fit is a transmission rate of 0.62 a day, $R_0 = 3.10$ against a truth of 2.5, with 5.5 initial exposed people. Its forecast is a peak of 6,192 beds on day 90.75, against a truth of 4,809 on day 100.5. The data have been fitted well, and the forecast is still far off. How far off could it have been?

A **parametric bootstrap** answers that. Chapter 13 resampled residuals; here we know the kind of noise (we made it), so we draw new data sets from the fitted curve with the same Poisson and lognormal noise, refit each one, and record the peak each refit predicts. With real data you would estimate the noise model from the residuals first. Each refit is just a choice of the best of the 2,911 grid runs, and a little algebra in the code makes all 300 refits one matrix product (`@`, numpy's matrix multiplication).

With data to day 45, the 90% range for the peak runs from 3,439 to 8,142 beds and from day 79 to day 117. With data to day 50 it is 4,531 to 7,939 beds, days 82 to 102. Only with data to day 80, when the peak is three weeks away, does it narrow to 4,079 to 5,556 beds, days 98 to 102. The true peak, 4,809 beds on day 100.5, lies inside every one of these ranges, but the best single estimate swings from 6,192 at day 50 to 4,531 at day 60 and 5,198 at day 70.

In [ ]:
#| fig-alt: "Three panels of forecast bed occupancy, made with data to days 50, 60 and 80. Each shows a shaded 90% band, a median line, a dashed truth curve and a dotted capacity line at 1,000. The band is very wide at day 50 and narrow at day 80."
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, upto in zip(axes, (50, 60, 80)):
    paths = occ.to_numpy()[:, picks_by_cut[upto]]
    lo, mid, hi = np.percentile(paths, [5, 50, 95], axis=1)
    ax.fill_between(occ.index, lo, hi, color="#56B4E9", alpha=0.5,
                    label="90% band")
    ax.plot(occ.index, mid, color="#0072B2", label="median forecast")
    ax.plot(beds.index, beds, color="#000000", ls="--", label="truth")
    ax.axhline(1000, color="grey", ls=":", lw=1)
    ax.axvline(upto, color="#D55E00", ls="-.", lw=1)
    ax.set(title=f"Data to day {upto}", xlabel="time (days)")
axes[0].set_ylabel("beds occupied (beds)")
axes[0].legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

Figure: Forecasts of beds occupied made with data to days 50, 60 and 80 (vertical lines), each with its 90% bootstrap band, the median forecast and the truth (dashed). Notice how wide the band is at day 50 and how quickly it narrows as the curve bends.

Why is the early forecast so uncertain? In the growth phase the data show one thing clearly: the growth rate. Many pairs of transmission rate and starting size give the same early curve, and a small change in the growth rate compounds into a large change in where the curve turns. This is the lesson of chapter 5's logistic exercise, now with numbers: **early growth carries little information about the peak**. Every forecast at day 50 agrees on one thing, though: beds will run out. Across the bootstrap, occupancy first passes 1,000 between about day 63 and day 77 (Try it 5). The decision about whether to act does not wait for the peak forecast to settle.

**Try it 5.**

- *Check.* What is the 90% range of $R_0$ among the refits with data to day 50, and with data to day 70?
- *Apply.* With data to day 50, find the 90% range for the day beds first exceed 1,000.
- *Extend.* Suppose the hospital share is not known: it could be 1%, 2% or 3%. Fit the grid for each share to the data to day 50. Can the data tell the shares apart, and what does each imply for the peak?

<details><summary>Worked answer</summary>

With data to day 50, $R_0$ runs from 2.4 to 4.3; with data to day 70, from 2.3 to 3.25. Beds first exceed 1,000 between day 63.0 and day 76.5 (median 68.5). With an unknown share, the three fits are equally good (sums of squared errors of 75.9, 75.6 and 75.7), with $R_0$ of 3.05, 3.10 and 3.00, yet they forecast peaks of 3,046, 6,192 and 8,987 beds. Early admissions identify the hospital share times the number infected, not either one, just as the waiting list identified only the gap between referrals and capacity (chapter 13). A survey of infections, or a hospital share measured in another region, would break the tie.
</details>

In [ ]:
r0_50 = 5 * BB.ravel()[picks_by_cut[50]]
r0_70 = 5 * BB.ravel()[picks_by_cut[70]]
print("check: 90% range of R0, data to day 50:",
      np.percentile(r0_50, [5, 95]).round(2), "| day 70:",
      np.percentile(r0_70, [5, 95]).round(2))
first_full = (occ > 1000).idxmax().to_numpy()        # first day over 1,000
print("apply: day beds first exceed 1,000, data to day 50, 90% range:",
      np.percentile(first_full[picks_by_cut[50]], [5, 50, 95]))
# extend: hospital share unknown as well (1%, 2% or 3%)
SHARES = (0.01, 0.02, 0.03)
# dict(r, key=value) copies the grid row r and adds the share
rows = [dict(r, hospital_share=h) for h in SHARES
        for r in grid.to_dict("records")]
runs3 = epi.run_batch(pd.DataFrame(rows), T=250,
                      include=["Admitted so far", "Beds occupied"])
cum3 = runs3[runs3.variable == "Admitted so far"].pivot(
    index="time", columns="run", values="value")
occ3 = runs3[runs3.variable == "Beds occupied"].pivot(
    index="time", columns="run", values="value")
daily3 = np.diff(cum3.loc[np.arange(0.0, 51)].to_numpy(), axis=0)
sse3 = ((daily3 - y[:50, None]) ** 2).sum(axis=0)
n = len(grid)
for b, h in enumerate(SHARES):
    block = np.arange(b * n, (b + 1) * n)    # the runs with this share
    k = block[sse3[block].argmin()]
    print(f"extend: share {h:.0%}: best SSE {sse3[k]:,.1f}, R0 "
          f"{5 * BB.ravel()[k - b * n]:.2f}, peak beds "
          f"{occ3[k].max():,.0f} on day {occ3[k].idxmax()}")

## 6. Policy experiments and robustness

The authority has two kinds of lever. **Contact measures** (closures, distancing, working from home) cut the contact multiplier for a period. **Surge capacity** adds beds, with a delay to build them. We judge each policy by patient-days without a bed, and also report the peak and the number ever infected. Chapter 15 sets out the method: policies as parameter settings, judged on several measures, then tested against what we do not know.

Start with timing and length. Halving contacts from day 50 gives an effective $R$ of 1.25: the epidemic still grows, but slowly. What happens when the measures are lifted?

In [ ]:
#| fig-alt: "Beds occupied over 400 days under four policies. With no measures one hump near 4,800 around day 100. Halving contacts from day 50 for 60 days delays the hump to around day 150 with a peak near 4,000. For 120 days the rebound comes later still. Held to day 400, beds stay low."
POLICIES = {
    "no measures": {},
    "halve contacts, days 50 to 110": dict(contact_reduction=0.5),
    "halve contacts, days 50 to 170": dict(contact_reduction=0.5,
                                           measures_length=120),
    "halve contacts from day 50 on": dict(contact_reduction=0.5,
                                          measures_length=400)}
fig, ax = plt.subplots()
for label, kw in POLICIES.items():
    d = epi.run(T=400, **kw)
    b = d["Beds occupied"]
    b.plot(ax=ax, label=label)
    print(f"{label:<31}: peak {b.max():5,.0f} on day {b.idxmax():6.2f}; "
          f"ever infected {1e6 - d['Susceptible'].iloc[-1]:7,.0f}; "
          f"patient-days without a bed "
          f"{d['Patients without a bed'].sum() * 0.25:7,.0f}")
ax.axhline(1000, color="grey", ls=":", lw=1)
ax.set(title="Measures that are lifted bring a rebound",
       xlabel="time (days)", ylabel="beds occupied (beds)")
ax.legend(fontsize=8)
plt.show()

Figure: Beds occupied under four contact policies, all halving contacts from day 50. Notice that lifting the measures brings back the epidemic: the hump is postponed rather than removed, unless the measures stay in place.

Halving contacts for 60 days only postpones the peak, from day 100.5 to day 151.25, and lowers it a little, from 4,809 to 4,019 beds. When the measures end on day 110, most people are still susceptible, effective $R$ jumps back to about 2.5 times the susceptible share, and the epidemic **rebounds**. Keeping the measures for 120 days does better (2,398 beds on day 213.25), and keeping them to the end of the run holds occupancy below capacity, with a peak of 571 beds and 386,448 people infected by day 400.

A measure that is lifted too early buys time, and time is only worth something if it is used: to build beds, to wait for a vaccine (outside our boundary), or to lift the measures when they will do least harm. A sweep over start day and length, 255 runs in one `run_batch` call, shows where the good settings lie.

In [ ]:
#| fig-alt: "A heat map of patient-days without a bed, with the day measures start (30 to 110) across and their length (30 to 240 days) up. Light cells, with few patient-days short, form a band for starts around days 55 to 70; the earlier the start, the longer the measures must last."
starts = np.arange(30, 111, 5)
lengths = np.arange(30, 241, 15)
S_, L_ = np.meshgrid(starts, lengths)
sweep = pd.DataFrame({"contact_reduction": 0.5,
                      "measures_start": S_.ravel(),
                      "measures_length": L_.ravel()})
res = epi.run_batch(sweep, T=400, include=["Patients without a bed"])
short = (res.groupby("run")["value"].sum() * 0.25).to_numpy().reshape(
    S_.shape)
k = np.unravel_index(short.argmin(), short.shape)   # row and column of min
print(f"least shortage: start day {S_[k]}, length {L_[k]} days, "
      f"{short[k]:,.0f} patient-days; {(short < 1).sum()} of {short.size} "
      f"settings have none")
none = short < 1
print(f"  they start on days {S_[none].min()} to {S_[none].max()} and last "
      f"{L_[none].min()} to {L_[none].max()} days")
for st in (50, 70, 90):
    col = list(starts).index(st)
    print(f"start day {st}: length 60 gives {short[2, col]:7,.0f}, "
          f"120 gives {short[6, col]:7,.0f}, 240 gives "
          f"{short[-1, col]:7,.0f} patient-days without a bed")
fig, ax = plt.subplots(figsize=(7, 4))
# pcolormesh: a coloured grid with one cell per (start, length) setting
im = ax.pcolormesh(S_, L_, short / 1000, cmap="viridis_r", shading="auto")
fig.colorbar(im, ax=ax, label="patient-days without a bed (thousands)")
ax.set(title="When to start and how long to keep measures",
       xlabel="day measures start (day)", ylabel="length of measures (days)")
plt.show()

Figure: Patient-days without a bed for every start day (across) and length (up) of measures that halve contacts. Notice the light band of good settings: early starts need long measures, while late starts gain nothing from extra length.

Only 11 of the 255 settings avoid any shortage, all starting on day 55 or 60 and lasting 150 days or more. Starting on day 50, 60 days of measures leave 88,118 patient-days without a bed and 240 days leave 16. Starting on day 70, the length hardly matters after 60 days (16,670 against 13,788 patient-days): measures that begin near the peak flatten it, and the epidemic burns out while they last. Starting on day 90 is too late at any length (81,384). So **earlier is better only when the measures last**; a short early measure can be worse than a short later one. The best timing depends on knowing when the peak would come, which section 5 showed we do not know well at day 50.

Now surge capacity: 1,000 extra beds, built over 14 days from day 60, alone and together with a smaller but lasting cut in contacts of 30%.

In [ ]:
#| fig-alt: "Beds occupied and bed capacity over 300 days. One capacity line stays at 1,000; the other climbs to 2,000 over two weeks from day 60. With no measures occupancy rises to about 4,800, far above both; with contacts cut by 30% for good the hump is about 2,400, lower and later, and only a little above the surged capacity."
SURGE = dict(surge_beds=1000, surge_start=60)
MODERATE = dict(contact_reduction=0.3, measures_length=400)
cases = {"no measures": {}, "surge only": SURGE,
         "30% fewer contacts, kept": MODERATE,
         "both": {**SURGE, **MODERATE}}
runs_s = {k: epi.run(T=300, **kw) for k, kw in cases.items()}
for label, d in runs_s.items():
    print(f"{label:<25}: peak {d['Beds occupied'].max():5,.0f}; days over "
          f"capacity {(d['Patients without a bed'] > 0).sum() * 0.25:5.1f};"
          f" patient-days without a bed "
          f"{d['Patients without a bed'].sum() * 0.25:7,.0f}")
fig, ax = plt.subplots()
runs_s["no measures"]["Beds occupied"].plot(ax=ax, label="occupied, "
                                            "no measures")
runs_s["30% fewer contacts, kept"]["Beds occupied"].plot(
    ax=ax, label="occupied, 30% fewer contacts kept")
runs_s["no measures"]["Bed capacity"].plot(ax=ax, color="#000000",
                                           ls=(0, (1, 3)),
                                           label="capacity, base")
runs_s["surge only"]["Bed capacity"].plot(ax=ax, color="#E69F00",
                                          ls="-.", label="capacity with "
                                          "surge")
ax.set(title="Surge beds and fewer contacts", xlabel="time (days)",
       ylabel="beds")
ax.legend(fontsize=8)
plt.show()

Figure: Beds occupied with no measures and with a lasting 30% cut in contacts, against the base capacity and the capacity with 1,000 surge beds. Notice that surge beds alone cannot meet an unmitigated peak, while a lasting cut brings the peak close to the surged capacity.

Surge beds alone cut patient-days without a bed from 107,563 to 62,912, but the unmitigated peak of 4,809 is still more than twice the surged capacity. A lasting 30% cut in contacts lowers the peak to 2,392 and gives 49,897 patient-days without a bed. Together they leave 6,680, and occupancy is above capacity for 26 days instead of 52.8. The two levers work on different parts of the structure: contact measures weaken the reinforcing loop, surge beds raise the ceiling the stock is compared with.

**Robustness.** Section 5 left $R_0$ uncertain, and the hospital share may be too. Chapter 15's test is to run every policy in every plausible world: here $R_0$ of 2, 2.5 and 3 crossed with hospital shares of 1.5%, 2% and 2.5%, 54 runs in one call.

In [ ]:
CANDIDATES = {
    "no measures": {},
    "halve contacts, days 50 to 110": dict(contact_reduction=0.5),
    "halve contacts, days 70 to 130": dict(contact_reduction=0.5,
                                           measures_start=70),
    "30% fewer contacts, kept": MODERATE,
    "surge only": SURGE,
    "30% fewer contacts, kept, and surge": {**SURGE, **MODERATE}}
rows, labels = [], []
for label, kw in CANDIDATES.items():
    for r0 in (2.0, 2.5, 3.0):
        for share in (0.015, 0.02, 0.025):
            # every default, then the policy, then this world's settings
            rows.append({**epi.params, **kw, "transmission_rate": r0 / 5,
                         "hospital_share": share})
            labels.append((label, r0, share))
scen = pd.DataFrame(rows)[list(epi.params)]      # every constant, in order
out = epi.run_batch(scen, T=400, include=["Patients without a bed"])
short = out.groupby("run")["value"].sum() * 0.25
scores = pd.DataFrame(labels, columns=["policy", "R0", "share"]).assign(
    short=short.to_numpy())
summary = scores.groupby("policy", sort=False)["short"].agg(
    ["mean", "max"]).round(-2)
by_r0 = scores[scores.share == 0.02].pivot(
    index="policy", columns="R0", values="short").round(-2)
print(summary.to_string())
print("\nhospital share 2%, by R0:")
print(by_r0.loc[list(CANDIDATES)].to_string())

Across the nine worlds, the combination of a lasting 30% cut and surge beds is best on average (17,300 patient-days without a bed) and in the worst case (70,700, against at least 122,600 for every other policy). The fixed 60-day measures are fragile. Halving contacts from day 70 is the best single policy if $R_0$ is 2.5 (16,700 patient-days with a 2% share) but poor if it is 2 or 3 (60,700 and 88,400), because its fixed dates hit the peak in only one of the worlds. With a 2% share, halving contacts from day 50 for 60 days never does better than 59,600. A policy tuned to the best estimate is not a robust policy; one that does not depend on the timing of the peak is.

**Try it 6.**

- *Check.* What effective $R$ do contact cuts of 50%, 60% and 70% give at the start of the epidemic?
- *Apply.* Cut contacts by 70% from day 50 to day 110. What is the peak, and when? How many people are still susceptible on day 110?
- *Extend.* With no contact measures, how many surge beds (from day 60, built over 14 days) are needed to avoid any patient going without a bed?

<details><summary>Worked answer</summary>

Effective $R$ is 1.25, 1.00 and 0.75: only a cut of more than 60% makes the epidemic shrink. A 70% cut for 60 days suppresses the epidemic, but 974,107 people are still susceptible when it ends, so the rebound is almost the full epidemic: a peak of 4,529 beds on day 177. Surge beds of 500, 1,000 and 2,000 leave 83,395, 62,912 and 30,648 patient-days without a bed; 4,000 extra beds leave none, a fivefold increase in capacity within weeks, which few regions could build.
</details>

In [ ]:
for cr in (0.5, 0.6, 0.7):
    print(f"check: contacts cut by {cr:.0%}: effective R at the start "
          f"{2.5 * (1 - cr):.2f}")
d = epi.run(T=400, contact_reduction=0.7, measures_start=50,
            measures_length=60)
b = d["Beds occupied"]
print(f"apply: 70% cut, days 50 to 110: peak {b.max():,.0f} on day "
      f"{b.idxmax()}; {d.loc[110.0, 'Susceptible']:,.0f} still susceptible "
      f"on day 110")
for sb in (500, 1000, 2000, 4000):
    d = epi.run(T=300, surge_beds=sb, surge_start=60)
    print(f"extend: surge {sb:>4} beds: patient-days without a bed "
          f"{d['Patients without a bed'].sum() * 0.25:7,.0f}")

## 7. The AI pass: every stage again, with AI

The case has now been through the four stages of the modelling process by hand. This section goes through them again with AI help, on the same model and data, under the contract of @sec-ai-partners. Each stage uses the kind of AI that fits it.

| Stage | Kind of AI | Task here | Check by code | Pattern from |
|---|---|---|---|---|
| Conceptualise | Generative | Causal links from a short case description | Links and signs against the spec; loops found | @sec-ai-conceptualise |
| Formulate | Reasoning | A review of the equations and decision rules | Each criticism quotes a real equation; buried numbers listed | @sec-ai-formulate |
| Test | Agent | Confirm the turning point; hunt for a failure | The claim rerun; every number traced to a tool | @sec-ai-test |
| Decide | Generative | A brief for the authority | Every number against the runs; certainty words flagged | @sec-ai-decide |

**Where the replies come from.** No cell here calls a model service. The replies in `data/ai/c14.json` were written for the book with claude-opus-5-5, with no live API call, and each record says so in its `how` field. Some contain faults planted on purpose, and the text says where. To ask a live model instead, install the client (`pip install anthropic`), set `ANTHROPIC_API_KEY` to your key and `LB_AI_LIVE=1`, and run the chapter. Requests with a recorded reply still replay; new ones go to the live model and are recorded as live calls.

The cell below notes the replay key of every request, for the record at the end, as @sec-ai-conceptualise does. It also defines `numbers`, which reads every number written in a text.

In [ ]:
# Every request has a replay key, a hash of everything sent. This small
# logger notes the keys each stage uses, for the record at the end.
import re as regex      # "re" already names effective R in section 4

KEYS, STAGE_KEYS = [], {}
_plain_key = ai.request_key


def logged_key(*args, **kw):
    key = _plain_key(*args, **kw)
    KEYS.append(key)
    return key


ai.request_key = logged_key


def numbers(text):
    """Every number written in a text, as floats ("4,809" is 4809)."""
    return [float(x.replace(",", ""))
            for x in regex.findall(r"\d[\d,]*(?:\.\d+)?", str(text))]

### Conceptualise: a generative draft of the links

**Stage task:** draft the causal links behind the reference mode. **Kind of AI:** generative. **Check:** every link and its sign against the chapter's own spec, then the loops, found by code. **What stays human:** the boundary, and which links to keep.

The prompt gives the case in five plain sentences and the 14 names of the sketch at the top of the chapter. Exposed and In hospital stand for their stages. It asks for links with signs, as JSON.

In [ ]:
VARIABLES = ["Susceptible", "Exposed", "Infectious", "Recovered",
             "In hospital", "Infection", "Becoming infectious",
             "Recovering at home", "Admissions", "Discharges",
             "Beds occupied", "Bed capacity", "Patients without a bed",
             "Contact multiplier"]
CASE = ("A respiratory infection spreads in a region of one million people. "
        "Infected people are not infectious for a few days, then infect "
        "others for about five days. A small share of them go to hospital "
        "and stay about ten days. The authority can cut contacts for a "
        "period and add beds. It fears that the beds will run out.")
CLD = f"""You are helping a system dynamics modeller. From the case below,
draft the causal links of a stock-and-flow model of it. Use only these
variable names: {", ".join(VARIABLES)}.
A flow raises the stock it fills (+) and lowers the stock it drains (-).
Return only JSON: {{"links": [[cause, effect, "+" or "-"], ...]}}

CASE: {CASE}"""
start = len(KEYS)
links = ai.ask_json(CLD, replay=REPLAY)["links"]
STAGE_KEYS["Conceptualise"] = KEYS[start:]
print(f"{len(links)} links drafted:")
print("; ".join(f"{c} -> {e} ({s})" for c, e, s in links))

The draft has 20 links. The reply was written for the book, with two faults planted that drafts of epidemic models often contain, and one link left out. The check reads the chapter's own links from the spec by code. Every name an equation reads is a cause of it, and a flow raises the stock it fills and lowers the one it drains. The sign of each link comes from nudging the cause by 1% at the base run's peak in beds and seeing which way the equation moves.

In [ ]:
def lumped(name):
    """A spec name in the case's words: "Exposed 2" is part of Exposed."""
    return regex.sub(r" [123]$", "", name)


SPEC = SEIR_STOCKS + SEIR_FLOWS + SEIR_VARS + HOSP_STOCKS + HOSP_FLOWS \
    + HOSP_VARS
NAME = {lb.name_key(e["name"]): e["name"] for e in SPEC}
ID = {e["id"]: e["name"] for e in SPEC}
# the state at the base run's peak in beds, to read each link's sign
STATE = {**epi.params, "TIME": beds.idxmax(), "DT": 0.25,
         **{lb.name_key(c): v for c, v in base.loc[beds.idxmax()].items()}}


def sign_of(eq, ref):
    """+ if the equation rises when ref rises by 1%, - if it falls."""
    f = lb.compile_expr(eq)
    change = f(dict(STATE, **{ref: 1.01 * STATE[ref]})) - f(STATE)
    return "+" if change > 0 else "-" if change < 0 else "0"


chapter = {}                     # (cause, effect) -> sign, read from the spec
for e in SEIR_FLOWS + SEIR_VARS + HOSP_FLOWS + HOSP_VARS:
    for ref in lb.compile_expr(e["eq"]).refs:
        chapter[lumped(NAME[ref]), lumped(e["name"])] = sign_of(e["eq"], ref)
for f in SEIR_FLOWS + HOSP_FLOWS:
    for end, sign in (("to", "+"), ("from", "-")):
        if end in f:
            chapter[lumped(f["name"]), lumped(ID[f[end]])] = sign
chapter = {(c, e): s for (c, e), s in chapter.items()
           if c != e and {c, e} <= set(VARIABLES)}
for c, e, s in links:
    if (c, e) not in chapter:
        print(f"NOT IN THE MODEL: {c} -> {e} ({s})")
    elif chapter[c, e] != s:
        print(f"POLARITY: {c} -> {e} drafted {s}, the model has "
              f"{chapter[c, e]}")
drafted = {(c, e) for c, e, _ in links}
print("in the model, not drafted:",
      sorted(set(chapter) - drafted) or "none")
print(f"{len(chapter)} links in the chapter's model, {len(links)} drafted")

The check finds all three. The draft gives `Contact multiplier -> Infection` a minus sign. Measures cut infection, but the multiplier is 1 minus the cut, so a higher multiplier means more infection. It adds `Beds occupied -> Contact multiplier`, people cutting their contacts as beds fill. That is real behaviour, but section 2 put it outside the boundary. And it misses `Admissions -> Infectious`: an admission also drains the infectious stock. The modeller decides each one, in a comment, and code finds the loops.

In [ ]:
def find_loops(edges):
    """Every simple feedback loop in [(cause, effect, sign)], with its
    polarity: R if it has an even number of - links, B if odd."""
    out = {}
    for c, e, s in edges:
        out.setdefault(c, []).append((e, s))
    loops = []

    def follow(start, here, path, signs):
        for nxt, s in out.get(here, []):
            if nxt == start:
                loops.append((path, "BR"[(signs + [s]).count("-") % 2 == 0]))
            elif nxt > start and nxt not in path:
                follow(start, nxt, path + [nxt], signs + [s])

    for start in sorted(out):
        follow(start, start, [start], [])
    return loops


# The modeller's decisions. The multiplier is 1 minus the cut, so a higher
# multiplier means more infection (+). People cutting contacts as beds fill
# is outside the boundary of section 2, so that link stays out. Admissions
# drain the infectious stock, a link the draft missed.
decided = [(c, e, "+" if (c, e) == ("Contact multiplier", "Infection")
            else s) for c, e, s in links
           if (c, e) != ("Beds occupied", "Contact multiplier")]
decided.append(("Admissions", "Infectious", "-"))
KEY_LOOPS = {"contagion": ({"Infectious", "Infection", "Exposed",
                            "Becoming infectious"}, "R"),
             "depletion": ({"Susceptible", "Infection"}, "B")}
for label, edges in (("draft, as decided", decided),
                     ("chapter's model", [(c, e, s) for (c, e), s
                                          in chapter.items()])):
    loops = find_loops(edges)
    found = {k: (set(p), pol) in [(set(q), r) for q, r in loops]
             for k, (p, pol) in KEY_LOOPS.items()}
    print(f"{label}: {len(loops)} loops, "
          f"{sum(p == 'R' for _, p in loops)} reinforcing; key loops "
          f"found: {found}")
cld_ok = all({(c, e): s for c, e, s in decided}.get(k) == s
             for k, s in chapter.items()) and len(decided) == len(chapter)
print("decided draft matches the chapter's model link for link:", cld_ok)

As decided, the draft matches the chapter's 20 links one for one and gives the same 6 loops, one of them reinforcing. Both loops of the dynamic hypothesis are among them: contagion (R) and depletion (B). The code checked the draft against the model; it cannot say whether the boundary is right. Leaving out people's own response to full hospitals was a judgement in section 2, and it stays the modeller's, as Try it 7 shows.

### Formulate: a reasoning model reviews the rules

**Stage task:** check that each decision rule reads only what its decision maker can see, that the units balance and that every number has a name. **Kind of AI:** a reasoning model (`reasoning=True`), as in @sec-ai-formulate. **Check:** every criticism quotes a real equation, and code lists the numbers inside equations on its own. **What stays human:** which criticisms to act on.

The prompt lists every equation with its units and says who decides what. The authority sets the measures and the surge beds, and sees only admissions and beds occupied.

In [ ]:
WHO = ("The regional health authority decides the contact measures "
       "(Contact reduction, Measures start, Measures length) and the surge "
       "beds (Surge beds, Surge start). Each day it sees admissions and "
       "beds occupied. It does not see infections, the susceptible share or "
       "effective R.")
EQS = {e["name"]: e["eq"] for e in SEIR_FLOWS + HOSP_FLOWS + SEIR_VARS
       + HOSP_VARS}
UNITS = {e["name"]: e["units"] for e in SPEC}
LISTING = "\n".join(f"{n} = {eq}  [{UNITS[n]}]" for n, eq in EQS.items())
REVIEW = f"""Review the equations and decision rules of this epidemic and
hospital-bed model before it is used to advise a health authority. Units
are in square brackets.

{LISTING}

Who decides what: {WHO}

For each decision rule, ask: does it use only information the authority
has? Do the units balance? Is a number inside an equation, or a constant,
without a stated source? Reply with one JSON object:
{{"criticisms": [{{"element": "...", "quote": "the exact part of its
equation", "kind": "information, units, constant or other", "why": "...",
"suggestion": "..."}}]}}"""
start = len(KEYS)
review = ai.ask_json(REVIEW, reasoning=True, replay=REPLAY)
STAGE_KEYS["Formulate"] = KEYS[start:]
crit = pd.DataFrame(review["criticisms"])
with pd.option_context("display.max_colwidth", 48, "display.width", 120):
    print(crit[["element", "kind", "quote"]].to_string())

Seven criticisms. The reply was written for the book, and one point in it is planted: it criticises an occupancy target that this model does not have. That is a typical slip of a model that knows how hospitals usually plan. Fluent criticism can point at nothing, so code checks that each one quotes its element's equation. It also lists the numbers written inside equations, which a review should name.

In [ ]:
def squash(s):
    return "".join(str(s).split())          # ignore spaces


def literals(eq):
    """Numbers written inside an equation, other than 0 and 1."""
    out = []

    def walk(node):
        if node[0] == "num" and node[1] not in (0, 1):
            out.append(node[1])
        for child in node[1:]:
            for c in (child if isinstance(child, list) else [child]):
                if isinstance(c, tuple):
                    walk(c)

    walk(lb.compile_expr(eq).ast)
    return out


crit["real"] = [squash(c["quote"]) in squash(EQS.get(c["element"], "-"))
                for c in review["criticisms"]]
print(crit[["element", "kind", "real"]].to_string())
quoted = numbers(" ".join(crit.loc[crit.real, "quote"]))
buried = [(n, v) for n, eq in EQS.items() if literals(eq)
          and lb.compile_expr(eq).refs for v in literals(eq)]
print("numbers inside equations:", buried)
print("of those, quoted by no real criticism:",
      [(n, v) for n, v in buried if v not in quoted] or "none")

Six criticisms quote real equations. The occupancy target does not, and is rejected. Code finds five numbers inside equations: the 2 in the two latent flows and the 3 in the three hospital flows. The reviewer named the 2 but missed the 3, the number of hospital stages, so the modeller adds that point. Each decision is written beside the criticism it answers.

In [ ]:
# The modeller's decision on each point, in the reviewer's order
crit["decision"] = [
    "accept: keep dates for section 6; test a trigger (Going further)",
    "accept: state the source of the 2%; carry 1% to 3% (section 5)",
    "accept: name the 2 as Latent stages, used in both latent flows",
    "note: units balance, no change",
    "note: an output for us, read by no rule",
    "reject: quotes an equation the model does not have",
    "defer: outcomes without a bed are outside the boundary"]
with pd.option_context("display.max_colwidth", 70, "display.width", 120):
    print(crit[["element", "real", "decision"]].to_string())
print("and, found by code: name the 3 as Hospital stages, used in the "
      "three stay flows")
review_ok = bool((crit.real == ~crit.decision.str.startswith(
    "reject")).all())
print("every point kept quotes a real equation, and only the rejected one "
      "does not:", review_ok)

The sharpest point is the first. Measures that start on a fixed date are a policy experiment, not a rule the authority could follow, since it sees beds, not the epidemic. We keep the dates for section 6 and leave a trigger rule to Going further and the mini-project. Which criticisms matter for this decision is the modeller's call, not the reviewer's.

### Test: an agent hunts for a failure

**Stage task:** confirm the turning point of section 4 across the range of $R_0$, and hunt for settings where the model breaks. **Kind of AI:** an agent that calls tools in a loop [@yao2022react], as in @sec-ai-test. **Check:** code reruns the failure the agent claims, traces every number in its report to a tool and tests the turning point against theory. **What stays human:** the fix, and whether the model is fit for use.

The tools are three plain functions. `simulate` reports the peak in beds, the people ever infected and any people created or lost. `turning_point` compares the day effective $R$ falls below 1 with the day people infected now peak. `halve_dt` repeats a run at smaller time steps. Their docstrings are all the agent sees of them.

In [ ]:
def facts_of(settings, dt=0.25):
    """The facts a tester needs from one run, rounded for reading."""
    d = epi.run(T=250, dt=dt, warn_clips=False, **settings)
    b = d["Beds occupied"]
    people = settings.get("population", 1e6)
    return {"peak beds": round(float(b.max())),
            "peak day": float(b.idxmax()),
            "ever infected": round(float(people - d["Susceptible"].iloc[-1])),
            "people created or lost": round(float(
                (d[living].sum(axis=1) - people).abs().max()), 1),
            "floor held": sorted(d.attrs.get("clipped", {}))}


def simulate(settings: dict) -> dict:
    """Run the epidemic model for 250 days with parameter settings ({} for
    the base case; names as in the equations, in lower case with
    underscores, e.g. {"latent_period": 2}). Returns the peak in beds
    occupied (beds) and its day, the people ever infected, the largest
    number of people created or lost (should be 0), and the stocks the
    floor at zero had to hold up."""
    return facts_of(settings)


def turning_point(transmission_rate: float) -> dict:
    """Run the base case for 400 days with a transmission rate (1/day; R0
    is 5 times it). Returns R0, the first day effective R is below 1, the
    day people infected now (E + I) peak, the people susceptible that
    day, and N / R0."""
    d = epi.run(T=400, transmission_rate=transmission_rate)
    e_i = d[["Exposed 1", "Exposed 2", "Infectious"]].sum(axis=1)
    below = d.index[d["Effective R"] < 1]
    return {"R0": round(5 * transmission_rate, 2),
            "effective R below 1 from day":
                float(below[0]) if len(below) else None,
            "E + I peaks on day": float(e_i.idxmax()),
            "susceptible then": round(float(d.loc[e_i.idxmax(),
                                                  "Susceptible"])),
            "N / R0": round(1e6 / (5 * transmission_rate))}


def halve_dt(settings: dict) -> dict:
    """The same run as simulate at dt 0.25, 0.125 and 0.0625 days: the peak
    in beds and the people created or lost, for each dt."""
    return {str(dt): {k: v for k, v in facts_of(settings, dt).items()
                      if k in ("peak beds", "people created or lost")}
            for dt in (0.25, 0.125, 0.0625)}


TOOLS = {f.__name__: f for f in (simulate, turning_point, halve_dt)}
print(turning_point(0.5))

In the base case both events fall on day 91.5, with 398,853 people susceptible against $N/R_0$ = 400,000, as section 4 found. The task asks the agent to test the theory at both ends of the range of $R_0$. It also asks it to look for failures inside the plausible ranges, at their ends and beyond.

In [ ]:
HUNT = """Test our epidemic and hospital-bed model, which runs with a time
step of 0.25 days. First, confirm or refute: people infected now (E + I)
peak on the day effective R falls below 1, with N / R0 people still
susceptible, for R0 from 1.5 to 4. Second, hunt for settings where people
are created or lost, or where the peak in beds depends on the time step.
Plausible ranges: transmission_rate 0.3 to 0.8 a day, latent_period 1 to
6 days, infectious_period 2 to 8 days, length_of_stay 3 to 20 days. Test
inside the ranges, at their ends and beyond. Finish with an answer that is
a JSON object: {"turning_point": "confirmed" or "refuted", "failure":
{"tool": ..., "args": ...} for the clearest failure, or null, "report":
three sentences for the modeller that quote only numbers the tools
returned}."""
hunt_start = len(KEYS)
hunt = ai.agent(HUNT, TOOLS, replay=REPLAY, max_steps=10)
STAGE_KEYS["Test"] = KEYS[hunt_start:]
ai.show(hunt, width=110)

The agent made seven tool calls. Its replies were written for the book, each conditioned on the real tool results shown. It confirmed the turning point at both ends: both events fall on day 197.0 at $R_0 = 1.5$ and on day 59.0 at $R_0 = 4$. Then it reasoned about the shortest time constant, a latent stage of half the latent period. At the low end of the range, 1 day, nobody is created. Beyond it, at 0.25 days, a stage lasts 0.125 days, shorter than the step of 0.25 days. The floor held both exposed stocks, and 77,953.1 people were created. At a step of 0.125 days nobody is, and the peak falls from 8,743 to 7,315 beds.

An agent's report is a claim. Code reruns the failure it names and tests the turning point against the theory of section 4. The `unsupported` check of @sec-ai-partners traces every number in the report.

In [ ]:
def unsupported(answer, steps, allowed=""):
    """Numbers in an answer that no tool result, tool argument or the
    allowed text contains (within 1%, so a rounded figure passes)."""
    seen = set(numbers(allowed))
    for s in steps:
        seen |= set(numbers(s.get("result", ""))) | set(
            numbers(s.get("args", "")))
    return sorted(n for n in numbers(answer)
                  if not any(abs(n - m) <= 0.01 * max(abs(m), 1)
                             for m in seen))


answer = hunt[-1]["answer"]
claim = answer["failure"]
rerun = TOOLS[claim["tool"]](**claim["args"])
saw = [s["result"] for s in hunt if s.get("tool") == claim["tool"]
       and s.get("args") == claim["args"]]
TOLD = HUNT + "".join(f.__doc__ for f in TOOLS.values())
print("claimed failure:", claim)
print("rerun gives what the agent saw:", bool(saw) and saw[0] == rerun)
print("numbers in the report without a source:",
      unsupported(answer["report"], hunt, TOLD) or "none")
turns = [s["result"] for s in hunt if s.get("tool") == "turning_point"]
for t in turns:                  # the theory of section 4, checked by code
    print(f"R0 {t['R0']}: E + I peak {t['E + I peaks on day']} against R "
          f"below 1 from {t['effective R below 1 from day']}; susceptible "
          f"{t['susceptible then'] / t['N / R0'] - 1:+.2%} from N / R0")
hunt_ok = (bool(saw) and saw[0] == rerun
           and rerun["people created or lost"] > 1
           and not unsupported(answer["report"], hunt, TOLD)
           and all(abs(t["E + I peaks on day"]
                       - t["effective R below 1 from day"]) <= 0.25
                   for t in turns))
print("all checks pass:", hunt_ok)

The rerun gives what the agent saw, and every number in its report has a source. Code confirms the theory: at both ends effective $R$ crosses 1 on the day $E + I$ peaks, with susceptibles within 0.2% of $N/R_0$. Inside the ranges the step matters little. At a transmission rate of 0.8 a day, the peak moves from 7,594 to 7,485 beds as `dt` falls to 0.0625 days. The failure lies outside the plausible range, so the model stands. The modeller writes the rule it implies into the model's notes: keep `dt` below the shortest stage time, here half the latent period.

### Decide: a drafted brief, checked and signed

**Stage task:** draft the brief for the plan that section 6 found most robust. **Kind of AI:** generative, as in @sec-ai-decide. **Check:** every number against a table built from the runs, and every word of certainty flagged. **What stays human:** whether each sentence is fair, and the signature.

The model sees only a table that code builds from the runs of section 6. It holds the central case for the four surge and contact policies, and the scores across the nine worlds.

In [ ]:
central = pd.DataFrame({k: {
    "peak beds": int(round(d["Beds occupied"].max())),
    "days over capacity": (d["Patients without a bed"] > 0).sum() * 0.25,
    "patient-days without a bed":
        int(round(d["Patients without a bed"].sum() * 0.25))}
    for k, d in runs_s.items()}).T
worlds = scores.groupby("policy", sort=False)["short"].agg(
    ["mean", "max"]).round(-2).astype(int)
TABLE = ("Central case (R0 2.5, hospital share 2%), 300 days:\n"
         + central.to_string() + "\n\nNine worlds (R0 2, 2.5 and 3; "
         "hospital share 1.5%, 2% and 2.5%), patient-days without a bed:\n"
         + worlds.to_string())
BRIEF = f"""Draft a brief of four or five short sentences for a regional
health authority that does not read code. The analysts recommend a lasting
30% cut in contacts with 1,000 surge beds built over 14 days from day 60.
Use only the numbers in the tables below, rounded as a careful analyst
would. Say what happens without action, what the plan buys, and how sure
we are.

{TABLE}"""
start = len(KEYS)
draft = ai.ask(BRIEF, replay=REPLAY)
STAGE_KEYS["Decide"] = KEYS[start:]
print(draft)

The reply was written for the book with two slips that drafts often contain. One is a number the model worked out for itself; the other is a word of certainty that no model run can give. The checks are those of @sec-ai-decide.

In [ ]:
CERTAIN = r"\b(will|guarantee\w*|certain\w*|definitely|ensure\w*|always|" \
          r"never|eliminat\w*|prove\w*)\b"


def check_brief(text, table, tol=0.03):
    """Each number in text against its nearest in the table: ok within
    tol, the rounding a careful analyst would use."""
    allowed = numbers(table)
    rows = []
    for v in numbers(text):
        near = min(allowed, key=lambda a: abs(a - v))
        rows.append({"in the draft": v, "nearest in the table": near,
                     "ok": abs(near - v) <= tol * max(abs(near), 1)})
    return pd.DataFrame(rows)


def certainty(text):
    """Sentences that claim more certainty than a model can give."""
    return [s.strip() for s in regex.split(r"(?<=\.)\s+", text)
            if regex.search(CERTAIN, s, regex.I)]


nums = check_brief(draft, BRIEF)
print(f"{nums.ok.sum()} of {len(nums)} numbers within 3%; failing:",
      nums.loc[~nums.ok, "in the draft"].tolist())
print("certainty to look at:", certainty(draft))

Of the 11 numbers in the draft, 10 are within 3% of the table. The 94 is not in it: the model worked out the cut from 107,563 to 6,680 itself, which is right but not its job. The certainty check flags two sentences that say the plan or the forecast "will" do something. A person edits both, and the record keeps the edits.

In [ ]:
# A person reads the flags, edits and signs; the edits are the record.
EDITS = [("they will cut patient-days without a bed by 94%, to about 6,700",
          "they would cut patient-days without a bed in the central case "
          "from about 108,000 to about 6,700"),
         ("the forecast will sharpen", "the forecast should sharpen")]
signed = draft
for old, new in EDITS:
    signed = signed.replace(old, new)
brief_ok = bool(check_brief(signed, BRIEF).ok.all() and not certainty(signed))
assert brief_ok, "the signed brief still has a flag"
print(signed, "\n\nSigned: the analyst (name), for the modelling team")

The `assert` reruns both checks on the signed text, so it cannot leave the notebook with a flag on it. The checks cannot see what a brief leaves out. The person who signs checks that the worst case is there, as it is here: 70,700 patient-days without a bed in the hardest of the nine worlds.

### The record

The table below gives each stage's replay keys, how and when each reply was made, whether the stage's check passed and what a person decided. It is built from `ai.records(REPLAY)` and the checks above.

In [ ]:
ai.request_key = _plain_key              # stop logging keys
CHECKS = {
    "Conceptualise": (cld_ok, "one polarity fixed, one link added, the "
                              "behaviour link left out"),
    "Formulate": (review_ok, "three accepted, two noted, one deferred, "
                             "one rejected"),
    "Test": (hunt_ok, "failure accepted: keep dt below the shortest stage"),
    "Decide": (brief_ok, "two edits, then signed by the analyst")}
rec = ai.records(REPLAY)
log = pd.DataFrame([(stage, k, rec[k]["how"].split(" (")[0], rec[k]["date"],
                     CHECKS[stage][0], CHECKS[stage][1])
                    for stage, keys in STAGE_KEYS.items() for k in keys],
                   columns=["stage", "key", "how", "date", "check passed",
                            "human decision"])
with pd.option_context("display.max_colwidth", 60, "display.width", 160):
    print(log.to_string(index=False))

Eleven records: one draft, one review, eight agent steps and one brief. All were written for the book, every check passed, and every stage ends with a person's decision.

**Try it 7.**

- *Check.* Which two drafted links did the check flag, and what did the modeller decide about each?
- *Apply.* Call the agent's `turning_point` tool yourself with a transmission rate of 0.7 a day ($R_0 = 3.5$). Does the turn come where section 4 says it should?
- *Extend.* Put the drafted link `Beds occupied -> Contact multiplier` back and find the loops again. What new loop appears, with what polarity, and where does the chapter discuss it?

<details><summary>Worked answer</summary>

*Check.* `Contact multiplier -> Infection (-)` has a reversed sign, corrected to +. `Beds occupied -> Contact multiplier (-)` is left out, because it lies outside the boundary of section 2. The missing `Admissions -> Infectious` is a gap, not a flagged link; the check lists it separately.

*Apply.* Yes. Effective $R$ falls below 1 on day 66.0, the day $E + I$ peaks, with 283,264 people susceptible against $N/R_0$ = 285,714, within 1%.

*Extend.* One new loop, through eight variables from Admissions to Infectious by way of Beds occupied and Contact multiplier. It is balancing: more infectious people fill the beds, which cuts contacts and so infection. This is the behaviour that section 2 left out and Going further suggests adding. The loop finder cannot say whether people really behave this way; that needs evidence.

</details>

In [ ]:
print("check:", [(c, e, s) for c, e, s in links
                 if chapter.get((c, e)) != s])
print("apply:", turning_point(0.7))
with_behaviour = decided + [("Beds occupied", "Contact multiplier", "-")]
print("extend:", [(" -> ".join(p), pol) for p, pol
                  in find_loops(with_behaviour) if "Contact multiplier" in p])

## Tell the decision maker

- **Beds will run out.** Every forecast consistent with the admissions so far has occupancy passing the 1,000 available beds, most likely between days 63 and 77. Without measures the peak is likely to be around 5,000 beds near day 100, but on today's data it could be anywhere from about 4,500 to 7,900 beds, between days 82 and 102. The forecast will sharpen quickly over the next three to four weeks.
- **Temporary measures postpone; lasting ones protect.** Halving contacts for 60 days and then lifting them moves the peak about seven weeks later and lowers it by only a sixth. Measures help most when they are kept until something else changes: more beds, a vaccine, or an epidemic that has passed its peak.
- **Timing by the calendar is fragile.** A start date that works if the epidemic grows as we expect can fail if it grows faster or slower. Prefer policies that work across that range.
- **Combine the levers.** A lasting cut in contacts of about 30% with 1,000 surge beds built from day 60 cut patient-days without a bed by 94% in our central case (from 107,563 to 6,680), and did best in every one of the nine scenarios we tested.
- **How AI was used.** Parts of this analysis were drafted, reviewed or tested with an AI model (section 7). Every number here comes from the model runs, and a member of the team checked and signed each step.

## Check it

Before you show a decision maker an epidemic model:

- **People add up.** Write the conservation test as a function and run it on every policy, not just the base case.
- **The turning point is where theory puts it.** People infected now should peak when $R_e = 1$, that is when $S/N = 1/R_0$.
- **The step is small enough.** Halve `dt` and check that the peak in beds and its timing barely move.
- **AI help is labelled and traced.** Every recorded reply says how and when it was made, and every number in an agent's report comes from a tool.

The function below wraps the conservation test so it can be run on every policy.

In [ ]:
def check_people(run, population=1e6, tol=1e-3):
    """Every person is in exactly one living stock at every step."""
    gap = (run[living].sum(axis=1) - population).abs().max()
    assert gap < tol, f"people created or lost: {gap:.3g}"
    return gap


for label, kw in POLICIES.items():
    print(f"{label:<31}: drift {check_people(epi.run(T=400, **kw)):.1e}")
assert all(r["how"] and r["date"] for r in ai.records(REPLAY).values())
assert not unsupported(hunt[-1]["answer"]["report"], hunt, TOLD)

The drift is about $10^{-9}$ people under every policy, so the `STEP` switches and the staged delays create or lose nobody. Every AI record is labelled, and the agent's report traces to its tools.

## Going further

- **Behaviour that responds.** Make the contact multiplier fall as beds fill, through an effect function (chapter 7) of occupancy relative to capacity, perhaps with a perception delay (chapter 6). The rebound becomes a series of waves, and the timing of measures becomes a decision rule rather than a date.
- **Triggers rather than dates.** Start measures when occupancy passes a threshold and stop them when it falls below another. Rules like this are more robust to an uncertain $R_0$ than fixed dates; test them with the scenario grid of section 6.
- **More structure.** Age groups (an ageing chain across the population, not just the stay), waning immunity (a flow from Recovered back to Susceptible) and vaccination (a flow from Susceptible to Recovered) are each a few lines of spec. @homer2006 survey system dynamics models in public health.
- **Fitting with a likelihood.** Daily counts are better described by a Poisson or negative binomial likelihood than by squared errors, which give the largest days the most weight. Maximising a likelihood over the same grid is a short change to the payoff.
- **Networks and individuals.** A compartmental model assumes everyone mixes with everyone. @rahmandad2008 compare such models with agent-based models on contact networks and show when the difference matters.

## Mini-project

Turn the model into a decision rule and test it.

1. Add a trigger: contacts are cut by 50% while beds occupied exceed 800 and restored when they fall below 400. You need a stock that remembers whether the measures are on, switched by flows that use `IF`.
2. Calibrate to the admissions up to day 50 and run the trigger rule on all bootstrap refits. Report the 90% range of patient-days without a bed and the number of days with measures in place.
3. Compare with the best fixed-date policy of section 6 across the same refits and the nine robustness worlds.
4. Write a half-page brief for the regional health authority, with one figure, recommending a rule and saying what would change your mind.

## Further reading

- @sterman2000, chapter 9, treats epidemics as S-shaped growth, including the threshold and the overshoot.
- @homer2006 on system dynamics in public health, and @rahmandad2008 on when aggregate epidemic models hold up against network models.
- @bass1969 on the diffusion model behind the infection flow, and @hines_molecules for the molecules used here.
- @kermack1927 is the origin of the threshold result and the final-size equation of section 4.
- @hosseinichimeh2024 on drafting causal loop diagrams with language models, and @akhavan2024 on using them in modelling with care.
- @yao2022react on the reason-then-act loop of the agent in section 7, and @ji2023hallucination on why fluent drafts state things that are not so.